# ML-09 — Validation and Research Claim Audit

This notebook audits two research-paper findings and then turns the same lens on the Week-5 growth-prediction model.


## 1. Two paper findings + my methodology questions

I picked two strong findings from *The State of AI-Driven SEO, March 2026* pdf:

1. **Finding #4 — The Freshness Multiplier**  
   Claim: "365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions."  
   - *Where does the label come from?* "Refreshed" is defined by `days_since_update <= 30`. The outcome is FlyRank's composite **health score** (impressions 30 pts + position 30 pts + CTR 20 pts + scroll depth 20 pts).  
   - *Does the validation design carry the claim?* Probably not as strongly as the headline suggests. The refresh group is self-selected (someone chose to update those pages). Health score partly reuses impressions and position, so a visibility lift can mechanically lift the score. The `361+` freshness bucket is unstable: 283 growing vs only 1 declining page. This is observational and directional, not causal proof that refreshing *causes* growth.

2. **Finding #8 — The Age-Freshness Matrix**  
   Claim: "Old content that gets refreshed performs nearly as well as new content" (365+ × 0-30d fresh health = 44.62 vs 31-90d age × 0-30d fresh health = 44.12).  
   - *Where does the label come from?* Same composite health score, split by age tier and freshness tier.  
   - *Does the validation design carry the claim?* The matrix has strong survivor bias in the `365+ × 361+` cell and very small counts in several corners. The comparison mixes cross-sectional age with a refresh event that is not randomized. The safer read is: *among currently active older pages, recently refreshed ones look healthier* — not that refresh erases age effects for every page.

Both findings are presented responsibly in the paper (they note unstable buckets and composite limits), but the headline numbers could be quoted out of context.


In [1]:
# Print the two findings as a compact checklist
paper_findings = [
    {
        'id': 'Finding #4',
        'name': 'The Freshness Multiplier',
        'headline': '365+ day content refreshed within 30 days shows 3.2x health boost and 57x more impressions.',
        'label': 'Refresh = days_since_update <= 30; outcome = composite health score',
        'question': 'Is the refresh group self-selected? Does health score reuse impressions/position? Is the 361+ bucket unstable?'
    },
    {
        'id': 'Finding #8',
        'name': 'The Age-Freshness Matrix',
        'headline': 'Old + refreshed content performs nearly as well as young + fresh content.',
        'label': 'Health score by age tier x freshness tier',
        'question': 'Are small cells and survivor bias in the 365+ x 361+ bucket hiding the true age effect? Is refresh randomized?'
    }
]

for f in paper_findings:
    print(f"{f['id']} — {f['name']}")
    print(f"  Headline: {f['headline']}")
    print(f"  Label/source: {f['label']}")
    print(f"  Methodology question: {f['question']}\n")


Finding #4 — The Freshness Multiplier
  Headline: 365+ day content refreshed within 30 days shows 3.2x health boost and 57x more impressions.
  Label/source: Refresh = days_since_update <= 30; outcome = composite health score
  Methodology question: Is the refresh group self-selected? Does health score reuse impressions/position? Is the 361+ bucket unstable?

Finding #8 — The Age-Freshness Matrix
  Headline: Old + refreshed content performs nearly as well as young + fresh content.
  Label/source: Health score by age tier x freshness tier
  Methodology question: Are small cells and survivor bias in the 365+ x 361+ bucket hiding the true age effect? Is refresh randomized?



## 2. My model under an honest split (before/after)

The Week-5 notebook already moved to a 5-fold grouped CV split by `client_hash_id`. Here I compare that honest number with a simpler **random stratified split** that ignores clients. The gap between the two is how much the model was memorizing client-specific patterns.

The "before" number is the random split. The "after" number is the grouped CV mean from `w05_model_metrics.json`.


In [2]:
%pip install -q duckdb pandas scikit-learn matplotlib

import os
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import duckdb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

# Load cached feature vector from Week 5
cache_path = Path.cwd().parent / 'outputs' / 'model_feature_vector.csv'
feature_vector = pd.read_csv(cache_path)
print(f'Loaded feature vector: {len(feature_vector):,} rows')
print(f'Overall base growth rate: {feature_vector["growth_label"].mean():.3%}')

# Feature engineering (same as Week 5)
df = feature_vector.copy()
df['main_intent'] = df['main_intent'].fillna('unknown')
df['recent_share'] = (df['gsc_impressions_last_30d'] / df['gsc_impressions_90d']).clip(0, 1).fillna(0)
df['log_gsc_impressions_90d'] = np.log1p(df['gsc_impressions_90d'])
df['log_gsc_impressions_last_30d'] = np.log1p(df['gsc_impressions_last_30d'])
df['momentum_x_volume'] = df['recent_share'] * df['log_gsc_impressions_90d']

numeric_features = [
    'gsc_impressions_90d', 'gsc_avg_position_90d', 'gsc_clicks_90d', 'gsc_ctr_90d',
    'sessions_organic_90d', 'days_with_impressions_90d', 'gsc_impressions_last_30d',
    'search_volume', 'competition', 'cpc', 'word_count', 'content_age_days',
    'recent_share', 'log_gsc_impressions_90d', 'log_gsc_impressions_last_30d', 'momentum_x_volume'
]
categorical_features = ['content_type', 'main_intent']
df = pd.get_dummies(df, columns=categorical_features, prefix=categorical_features, dtype=int)
feature_cols = numeric_features + [
    c for c in df.columns
    if c.startswith('content_type_') or c.startswith('main_intent_')
]

X = df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
y = df['growth_label'].values

# Metric helpers
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def evaluate(name, y_true, scores):
    return {
        'model': name,
        'precision_at_10': precision_at_k(y_true, scores, k=10),
        'precision_at_50': precision_at_k(y_true, scores, k=50),
        'precision_at_100': precision_at_k(y_true, scores, k=100),
        'roc_auc': roc_auc_score(y_true, scores),
        'average_precision': average_precision_score(y_true, scores),
    }

# BEFORE: random stratified split (ignores clients)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

random_results = []

# Baseline rule on random test set
test_df = df.iloc[X_test.index].copy()
baseline_score = np.where(
    test_df['gsc_impressions_90d'] >= 500,
    test_df['recent_share'] * np.log1p(test_df['gsc_impressions_90d']),
    0.0
)
random_results.append(evaluate('baseline_rule', y_test, baseline_score))

# Gradient boosting on random split
gb_random = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
)
sample_weight = compute_sample_weight('balanced', y_train)
gb_random.fit(X_train, y_train, sample_weight=sample_weight)
gb_random_score = gb_random.predict_proba(X_test)[:, 1]
random_results.append(evaluate('gradient_boosting', y_test, gb_random_score))

random_df = pd.DataFrame(random_results)
print('\nBEFORE — Random stratified split (clients ignored):')
print(random_df.round(4).to_string(index=False))

# AFTER — load honest grouped CV from Week 5
w05_metrics_path = Path.cwd().parent / 'outputs' / 'w05_model_metrics.json'
with open(w05_metrics_path) as f:
    w05 = json.load(f)

cv_df = pd.DataFrame(w05['cv_summary'])
honest_df = cv_df[cv_df['model'].isin(['baseline_rule', 'gradient_boosting'])].copy()
honest_df['split'] = 'grouped_cv'
random_df['split'] = 'random_split'

comparison = pd.concat([
    random_df[['model', 'split', 'precision_at_10', 'precision_at_50', 'precision_at_100', 'roc_auc', 'average_precision']],
    honest_df[['model', 'split', 'precision_at_10', 'precision_at_50', 'precision_at_100', 'roc_auc', 'average_precision']]
], ignore_index=True)

print('\nBEFORE / AFTER comparison:')
print(comparison.round(4).to_string(index=False))

# Save
out_path = Path.cwd().parent / 'outputs' / 'w06_honest_split_comparison.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump({
        'random_split': random_df.to_dict(orient='records'),
        'grouped_cv': honest_df.to_dict(orient='records'),
        'timestamp': datetime.now(timezone.utc).isoformat()
    }, f, indent=2)
print(f'\nSaved: {out_path}')


Note: you may need to restart the kernel to use updated packages.


Loaded feature vector: 67,478 rows
Overall base growth rate: 24.731%



BEFORE — Random stratified split (clients ignored):
            model  precision_at_10  precision_at_50  precision_at_100  roc_auc  average_precision
    baseline_rule              0.4             0.44              0.44   0.5876              0.340
gradient_boosting              0.9             0.92              0.91   0.8211              0.648

BEFORE / AFTER comparison:
            model        split  precision_at_10  precision_at_50  precision_at_100  roc_auc  average_precision
    baseline_rule random_split             0.40            0.440             0.440   0.5876             0.3400
gradient_boosting random_split             0.90            0.920             0.910   0.8211             0.6480
gradient_boosting   grouped_cv             0.92            0.796             0.728   0.7305             0.4829
    baseline_rule   grouped_cv             0.50            0.428             0.464   0.5993             0.3512

Saved: /home/vincentoei/projects/flyrank-ml-internship-starter/work/o

## 3. Leakage audit

Repeat the leakage hunt from Week 3 on the final Week-5 feature set. I list the honest features, the excluded columns, and then deliberately add the target-window impression count as a feature. If the score jumps toward perfect, the test harness is sensitive to leakage and the feature belongs in the excluded bucket.


In [3]:
# Final honest features and excluded columns
print('Final feature count:', len(feature_cols))
print('Features:')
for c in feature_cols:
    print(f'  - {c}')

excluded = {
    'growth_label', 'gsc_impressions_next_30d', 'client_hash_id', 'content_hash_id',
    'trend_direction', 'trend_pct', 'content_updated_date', 'last_optimized_date',
    'health_score', 'priority_score', 'action_type'
}
print('\nExcluded / leakage-risk columns:')
for c in sorted(excluded):
    print(f'  - {c}')

used = set(feature_cols)
violations = used & excluded
print(f'\nViolations: {len(violations)}')
assert len(violations) == 0, f'Leakage risk: {violations}'

# Leakage attack: add the exact future impressions as a feature
X_honest = X.copy()
X_leaky = X.copy()
X_leaky['gsc_impressions_next_30d'] = df['gsc_impressions_next_30d'].fillna(0)

# Fast model for the attack
X_train_h, X_test_h, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, stratify=y, random_state=42)
X_train_l, X_test_l, _, _ = train_test_split(X_leaky, y, test_size=0.2, stratify=y, random_state=42)

attack_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
attack_model.fit(X_train_h, y_train)
honest_auc = roc_auc_score(y_test, attack_model.predict_proba(X_test_h)[:, 1])

attack_model.fit(X_train_l, y_train)
leaky_auc = roc_auc_score(y_test, attack_model.predict_proba(X_test_l)[:, 1])

print(f'\nHonest AUC (final features): {honest_auc:.4f}')
print(f'Leaky AUC (+ gsc_impressions_next_30d): {leaky_auc:.4f}')
print(f'AUC lift from leakage: {leaky_auc - honest_auc:+.4f}')
print('The leaky feature is removed. The honest feature set is kept.')


Final feature count: 24
Features:
  - gsc_impressions_90d
  - gsc_avg_position_90d
  - gsc_clicks_90d
  - gsc_ctr_90d
  - sessions_organic_90d
  - days_with_impressions_90d
  - gsc_impressions_last_30d
  - search_volume
  - competition
  - cpc
  - word_count
  - content_age_days
  - recent_share
  - log_gsc_impressions_90d
  - log_gsc_impressions_last_30d
  - momentum_x_volume
  - content_type_comparison article
  - content_type_feedly article
  - content_type_keyword article
  - main_intent_commercial
  - main_intent_informational
  - main_intent_navigational
  - main_intent_transactional
  - main_intent_unknown

Excluded / leakage-risk columns:
  - action_type
  - client_hash_id
  - content_hash_id
  - content_updated_date
  - growth_label
  - gsc_impressions_next_30d
  - health_score
  - last_optimized_date
  - priority_score
  - trend_direction
  - trend_pct

Violations: 0



Honest AUC (final features): 0.8012
Leaky AUC (+ gsc_impressions_next_30d): 0.9980
AUC lift from leakage: +0.1968
The leaky feature is removed. The honest feature set is kept.


## 4. Claim rewrite

One bold sentence from Week 5 could be misread as causal. Below is the unsafe version and the safe rewrite.


In [4]:
unsafe_claim = (
    "Gradient boosting predicts 30-day content growth with Precision@50 of 0.80, "
    "so prioritizing these pages will grow traffic."
)

safe_claim = (
    "In this observed dataset, pages ranked in the top 50 by the gradient-boosting score "
    "were more likely to show measured 30-day impression growth on held-out clients (CV Precision@50 = 0.80). "
    "The result is directional decision-support for prioritizing human review; it does not prove that promoting or editing "
    "these pages causes growth."
)

print('UNSAFE claim:')
print(unsafe_claim)
print('\nSAFE rewrite:')
print(safe_claim)

# Checklist
safe_words = ['observed', 'measured', 'directional', 'decision-support', 'more likely']
print('\nSafe-claim keywords present:')
for w in safe_words:
    print(f'  {w}: {w in safe_claim.lower()}')


UNSAFE claim:
Gradient boosting predicts 30-day content growth with Precision@50 of 0.80, so prioritizing these pages will grow traffic.

SAFE rewrite:
In this observed dataset, pages ranked in the top 50 by the gradient-boosting score were more likely to show measured 30-day impression growth on held-out clients (CV Precision@50 = 0.80). The result is directional decision-support for prioritizing human review; it does not prove that promoting or editing these pages causes growth.

Safe-claim keywords present:
  observed: True
  measured: True
  directional: True
  decision-support: True
  more likely: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] Two paper findings chosen and methodology questions framed constructively
- [x] My model re-run under an honest split with a before/after comparison
- [x] Leakage audit repeated on the final feature set
- [x] Boldest claim rewritten in safe language
- [x] Notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
